# NND Multi-Source PRS and symtoms data prep

## Purpose
Ingests multi-source Polygenic Risk Scores (PRS) and matches them with clinical metadata and longitudinal symptom tracking logs. It normalizes genetic scores, calculates adjusted $APOE$ variant traits, isolates historical pre-2022 patient batches, and dynamically models age-at-symptom timelines for downstream clinical profiling.

## Input
* `PATH_DISORDER_PRS` (`/mnt/../2_2025_output/`): Directory containing core disorder `.sscore` files.
* `PATH_NEUROPATH_PRS` (`/home/../output_neuropathological_traits`): Directory containing neuropathology `.sscore` files.
* `PATH_GSEM_PRS` (`/home/../output_gsem_factors`): Directory containing GSEM structural factors.
* `PATH_OTHER_TRAITS_PRS` (`/home/../other_traits`): Directory containing supplementary physiological traits.
* `PATH_APOE_DATA` (`/home/../test3.tsv`): Reference table containing raw `e2_dose` and `e4_dose` alleles.
* **NND Relational SQL Database:** PostgreSQL instance supplying patient demographics, clinical diagnostics, and raw longitudinal symptom history sheets.

> **Note:** The file paths listed above are environment placeholders. Replace them with your local directory mappings before running the pipeline.

## Process
1. **File Ingestion & Clinical Aggregation:** Automatically loops through all raw PRS directories to load genomic profiles, matching them against demographic age records and diagnostic categories pulled from the PostgreSQL instance.
2. **APOE Variant Correction:** Scales genetic scores into $Z$-scores and builds a consolidated Alzheimer's feature ($AD_{APOE}$) using targeted beta weights ($B_{e2} = -0.47$, $B_{e4} = 1.12$).
3. **Data Harmonization:** Aligns string definitions, resolves sender-receiver sample swaps, and filters out pediatric/adolescent cases (under 21) to generate the master core file (`merged_df`).
4. **Longitudinal Feature Pruning:** Slices unnecessary identifier and high-order ancestry variants from the core dataset to prevent downstream regression interference.
5. **Batch Filtering & Alignment:** Left-joins the longitudinal symptom data logs back to the genetic master file using baseline donor map keys, isolating historical patient records validated strictly prior to the `pre2022` threshold batch.
6. **Dynamic Timeline Modeling:** Back-calculates a dynamic `age_at_symptom` matrix variable for each logged clinical event by computing time deltas between a patient's file registry year and their true historical year of report.

## Output
* `output/merged_data.csv`: Unified master clinical-genetic dataframe (`merged_df`) ready for standalone population testing.
* `output/coombes_df.csv`: Final preprocessed, pruned longitudinal dataset containing baseline genetic scores mapped against age-at-symptom milestones, optimized for multivariate logistic regression and forest dot plotting.

In [ ]:
# ==============================================================================
# GLOBAL CONFIGURATION & DATA FLOW TOGGLES
# ==============================================================================

# File system paths for raw input datasets [placeholder, should be updated]
PATH_DISORDER_PRS     = "/mnt/../2_2025_output/"
PATH_NEUROPATH_PRS    = "/home/../output_neuropathological_traits"
PATH_GSEM_PRS         = "/home/../output_gsem_factors"
PATH_OTHER_TRAITS_PRS = "/home/../other_traits"
PATH_APOE_DATA        = "/home/../test3.tsv"

# Database connection details [Sensitive credentials removed]
DB_USER = ""
DB_NAME = ""
DB_HOST = ""
DB_PORT = ""
DB_TOKEN = ""
# OPTIONAL RUN CONFIGURATIONS
# ------------------------------------------------------------------------------
# Set the following booleans to True/False to control which inputs are loaded 
# and merged into your primary downstream analytical dataframe.
LOAD_NEUROPATH_PRS  = True   # Toggle processing of neuropathological traits
LOAD_GSEM_PRS       = True   # Toggle processing of GSEM factors
LOAD_OTHER_TRAITS   = True   # Toggle processing of other traits

# Standard set vs. Symptom-PRS interaction set toggle
# Toggle this list definition below based on your analytical requirements.
USE_SYMPTOM_INTERACTION_PRS = False




# 1. Data Prep: loading from sources <a name="1-data-preparation"></a>


In [ ]:
# ==============================================================================
# STAGE 0: IMPORTS
# ==============================================================================
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import glob
import psycopg2
import itertools
import numpy as np
from scipy.stats import zscore, mannwhitneyu
from statsmodels.stats import multitest
from statsmodels.stats.multitest import multipletests
from statannotations.Annotator import Annotator
import statsmodels.formula.api as smf
from scattermap import scattermap
from utils import replace_dict_overview, rename_dict




## Data loading from source

### Source 1: Core Disorders PRS

In [ ]:
# ==============================================================================
# STAGE 1: INGESTING AND FORMATTING MULTI-SOURCE PRS SCORES (.sscore)
# ==============================================================================


print(">>> Processing Disorder PRS...")
file_pattern = os.path.join(PATH_DISORDER_PRS, '*.sscore')
file_list = glob.glob(file_pattern)

prs_interest_disorder = []
dfs = []
for file in file_list:
    df = pd.read_csv(file, delimiter='\t')
    filename = os.path.basename(file).replace('.sscore', '')
    print(f"Loading: {filename}")
    df.rename(columns={'SCORE1_AVG': filename, 'IID': "donor_id"}, inplace=True)
    df = df[["donor_id", "PHENO1", filename]]
    prs_interest_disorder.append(filename)
    dfs.append(df)

disorder_prs = pd.concat(dfs, axis=1)
disorder_prs = disorder_prs.loc[:, ~disorder_prs.columns.duplicated()]
disorder_prs = disorder_prs[["donor_id"] + prs_interest_disorder]
disorder_prs['donor_id'] = 'NND ' + disorder_prs['donor_id']

print("Disorder PRS Shape:", disorder_prs.shape)
display(disorder_prs.head())


### Source 2: Neuropathological Traits (Optional)

In [ ]:

if LOAD_NEUROPATH_PRS:
    print("\n>>> Processing Neuropathological PRS...")
    file_pattern = os.path.join(PATH_NEUROPATH_PRS, '*.sscore')
    file_list = glob.glob(file_pattern)

    prs_np_interest = []
    dfs = []
    for file in file_list:
        df = pd.read_csv(file, delimiter='\t')
        filename = os.path.basename(file).replace('_prs.sscore', '')
        print(f"Loading: {filename}")
        df.rename(columns={'SCORE1_AVG': filename, 'IID': "donor_id"}, inplace=True)
        df = df[["donor_id", "PHENO1", filename]]
        prs_np_interest.append(filename)
        dfs.append(df)
        
    npprs_df = pd.concat(dfs, axis=1)
    npprs_df = npprs_df.loc[:, ~npprs_df.columns.duplicated()]
    npprs_df = npprs_df[["donor_id"] + prs_np_interest]
    npprs_df['donor_id'] = 'NND ' + npprs_df['donor_id']
    print("Neuropathological PRS Shape:", npprs_df.shape)
    display(npprs_df.head())


### Source 3: GSEM Factors (Optional)

In [ ]:

if LOAD_GSEM_PRS:
    print("\n>>> Processing GSEM Factors...")
    file_pattern = os.path.join(PATH_GSEM_PRS, '*_sbrc.txt_prs.sscore')
    file_list = glob.glob(file_pattern)

    prs_grotzinger = []
    dfs = []
    for file in file_list:
        df = pd.read_csv(file, delimiter='\t')
        filename = os.path.basename(file).replace('_sbrc.txt_prs.sscore', '')
        filename = os.path.basename(filename).replace('2025-05-03_', '')
        print(f"Loading: {filename}")
        df.rename(columns={'SCORE1_AVG': filename, 'IID': "donor_id"}, inplace=True)
        df = df[["donor_id", "PHENO1", filename]]
        prs_grotzinger.append(filename)
        dfs.append(df)
        
    grotzinger_df = pd.concat(dfs, axis=1)
    grotzinger_df = grotzinger_df.loc[:, ~grotzinger_df.columns.duplicated()]
    grotzinger_df = grotzinger_df[["donor_id"] + prs_grotzinger]
    grotzinger_df['donor_id'] = 'NND ' + grotzinger_df['donor_id']
    print("GSEM Factors Shape:", grotzinger_df.shape)
    display(grotzinger_df.head())


### Source 4: Other Traits (Optional)

In [ ]:

if LOAD_OTHER_TRAITS:
    print("\n>>> Processing Other Traits...")
    file_pattern = os.path.join(PATH_OTHER_TRAITS_PRS, '*.sscore')
    file_list = glob.glob(file_pattern)

    prs_traits = []
    dfs = []
    for file in file_list:
        df = pd.read_csv(file, delimiter='\t')
        filename = os.path.basename(file).replace('_prs.sscore', '')
        filename = os.path.basename(filename).replace('2025-05-03_', '')
        print(f"Loading: {filename}")
        df.rename(columns={'SCORE1_AVG': filename, 'IID': "donor_id"}, inplace=True)
        df = df[["donor_id", "PHENO1", filename]]
        prs_traits.append(filename)
        dfs.append(df)
        
    traits_df = pd.concat(dfs, axis=1)
    traits_df = traits_df.loc[:, ~traits_df.columns.duplicated()]
    traits_df = traits_df[["donor_id"] + prs_traits]
    traits_df['donor_id'] = 'NND ' + traits_df['donor_id']
    print("Other Traits Shape:", traits_df.shape)
    display(traits_df.head())



## Data Merging Pipeline for Multi-Source PRS

In [ ]:
# ==============================================================================
# STAGE 2: MERGING PIPELINE FOR MULTI-SOURCE PRS
# ==============================================================================
print("\n>>> Merging individual PRS dataframes...")
prs_df = disorder_prs.copy()

if LOAD_NEUROPATH_PRS and 'npprs_df' in globals() and npprs_df is not None:
    prs_df = prs_df.merge(npprs_df, on='donor_id', how='left')

if LOAD_GSEM_PRS and 'grotzinger_df' in globals() and grotzinger_df is not None:
    prs_df = prs_df.merge(grotzinger_df, on='donor_id', how='left')    

if LOAD_OTHER_TRAITS and 'traits_df' in globals() and traits_df is not None:
    prs_df = prs_df.merge(traits_df, on='donor_id', how='left') 

# Construct master lists of variables based on loaded datasets
prs_interest1 = prs_interest_disorder.copy()
if LOAD_NEUROPATH_PRS and 'prs_np_interest' in globals() and prs_np_interest:
    prs_interest1 += prs_np_interest
if LOAD_GSEM_PRS and 'prs_grotzinger' in globals() and prs_grotzinger:
    prs_interest1 += prs_grotzinger
if LOAD_OTHER_TRAITS and 'prs_traits' in globals() and prs_traits:
    prs_interest1 += prs_traits    


In [ ]:

# ==============================================================================
# STAGE 3: COLUMN RENAMING AND FIELD HARMONIZATION
# ==============================================================================


# Standardize columns using pre-defined map
prs_df = prs_df.rename(columns=rename_dict)
prs_interest1 = [rename_dict.get(col, col) for col in prs_interest1]

# Choose between analytical paths (Standard PRS vs. Interaction Traits)
if USE_SYMPTOM_INTERACTION_PRS:
    prs_interest = [
        'AD', 'ad_without_bigapoe_0.01', "PD", 'DLB', 'FTD',
        'Neurodegenerative', 'ALS', 'MS', 'MSA', 'MDD', 'BP', 'SCZ', 
        'risk_taking', 'intelligence', 'insomnia_schoeler_2023', 
        'sociability', 'neuroticism_hu'
    ]
else:
    prs_interest = [
        'AD', 'ad_without_bigapoe_0.01', "PD", 'DLB', 'FTD',
        'Neurodegenerative', 'ALS', 'MS', 'MSA', 'MDD', 'BP', 'SCZ'
    ]

# Keep only available harmonized traits
available_cols = ["donor_id"] + [col for col in prs_interest if col in prs_df.columns]
prs_df = prs_df[available_cols]

print("\nHarmonized Combined PRS:")
display(prs_df.head())
display(prs_interest)
print("Final Shape:", prs_df.shape)



## Data loading from NND (SQL database)

In [ ]:

# ==============================================================================
# STAGE 4: METADATA & CLINICAL SQL INGESTION
# ==============================================================================
print("\n>>> Extracting Metadata and Phenotypic parameters from DB...")
conn = psycopg2.connect(database=DB_NAME, user=DB_USER, host=DB_HOST, port=DB_PORT, password=DB_TOKEN)
cur = conn.cursor()

# Query 4a: Capture Personality Disorder information
personality_query = """
SELECT DISTINCT 
    dcd.donor_id AS donor_id,   
    cdm.name
FROM clinical_diagnosis_metadata cdm
JOIN donor_clinical_diagnosis dcd
    ON cdm.id = dcd.clinical_diagnosis_id
JOIN donor_main_diagnosis dmd
    ON dcd.donor_id = dmd.donor_id
JOIN main_diagnosis_metadata mdm
    ON dmd.nd_id = mdm.id
WHERE cdm.name IN (
    'antisocial personality disorder',
    'avoidant personality disorder',
    'borderline personality disorder',
    'histrionic personality disorder',
    'narcissistic personality disorder',
    'obsessive-compulsive personality disorder',
    'paranoid personality disorder',
    'personality disorder',
    'schizoid personality disorder',
    'schizotypal personality disorder'
)
AND mdm.nd = 'MDD'
ORDER BY dcd.donor_id ASC;
"""
cur.execute(personality_query)
personality_disorder = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
personality_disorder = personality_disorder.drop_duplicates(subset='donor_id', keep='first')
personality_disorder = personality_disorder[['donor_id', 'name']].reset_index(drop=True)
print("\n--- Personality Disorders extraction: ---")
display(personality_disorder.head())

# Query 4b: Demographic data (Age, Sex, Batch Year)
cur.execute("SELECT donor_id, age, most_likely_sex, file_year FROM general_information")
age_sex = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
print(f"Age & Sex metadata collected. Shape: {age_sex.shape}")

# Query 4c: Diagnosis information
cur.execute("SELECT donor_id, nd_full_names, nd_current FROM view_main_diagnosis")
diagnosis_sql = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
print(f"Main clinical diagnoses collected. Shape: {diagnosis_sql.shape}")

# Query 4d: Longitudinal clinical history & symptoms
symptoms_query = """
SELECT c.donor_id,
    c.year_of_report,
    b.batch,
    m.attribute,
    m.parameter,
    m.domain,
    m.grouping
FROM clinical_history_raw c
JOIN attributes_metadata m ON c.attribute_id = m.symptoms_id
JOIN batch_metadata b ON c.batch_id = b.id;
"""
cur.execute(symptoms_query)
symptoms = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
symptoms = symptoms.rename(columns={"batch": "batch_symptom"})
symptoms2022 = symptoms[symptoms['batch_symptom'] == 'pre2022']
print(f"Symptoms database pull finished. Unique pre-2022 patient count: {symptoms2022['donor_id'].unique().shape[0]}")

# Query 4e: Genetic mutation info for Frontotemporal Dementia (FTD) cases
cur.execute("SELECT * FROM FTD_genetic_diagnosis")
ftdmutation_sql = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])

# Process FTD mutation records
ftdmutation_sql["genetic_mutation"] = ftdmutation_sql.apply(
    lambda row: "unknown" if pd.isna(row["genetic_diagnosis"]) else 
                f"{row['genetic_diagnosis']}_{row['found']}" if not pd.isna(row["found"]) else 
                row["genetic_diagnosis"],
    axis=1
)
string_list = ['GRN_False', 'C9orf72_False', 'MAPT_False', 'Chr_False', 'Chr3_False', 'PSEN_False', 'PSEN1_False']
ftdmutation_sql["genetic_mutation"] = ftdmutation_sql["genetic_mutation"].apply(lambda x: "absent" if x in string_list else x)
ftdmutation_sql = ftdmutation_sql.drop(columns=["alteration"], errors="ignore")
ftdmutation_sql = ftdmutation_sql.drop_duplicates(subset=["donor_id", "found", "genetic_mutation"])

# Consolidate clinical entries: prioritize keeping actual mutations over "absent" values.
ftdmutation_sql = (
    ftdmutation_sql
    .groupby("donor_id", group_keys=False)
    .apply(lambda g: g if (g["genetic_mutation"] == "absent").all() else g[g["genetic_mutation"] != "absent"])
)
ftdmutation_sql = (
    ftdmutation_sql.groupby("donor_id", as_index=False)
    .agg({
        "genetic_diagnosis": lambda x: ",".join(sorted(x.unique())),
        "genetic_mutation": lambda x: ",".join(sorted(x.unique())),
        "found": "any"
    })
)
print("\n--- FTD genetic status summaries: ---")
display(ftdmutation_sql['genetic_mutation'].value_counts())



## Data merging: preparing final dataframe for downstream analysis

In [ ]:

# ==============================================================================
# STAGE 5: COMPREHENSIVE RECONCILIATION AND ALIGNMENT (MERGING DATABASE DATA)
# ==============================================================================
print("\n>>> Executing core database reconciliation and clinical-PRS merges...")

# Left joining clinical tables to the genetic-PRS master file
merged_df = diagnosis_sql.merge(prs_df, on='donor_id', how='right')
merged_df = merged_df.merge(age_sex, on='donor_id', how='right')
merged_df = merged_df.dropna(subset=['nd_current'])

# Incorporate ancestry vectors and mutation diagnostic statuses
if 'eigenvec_df' in globals() and eigenvec_df is not None:
    merged_df = merged_df.merge(eigenvec_df, on='donor_id', how='left')

merged_df = merged_df.merge(ftdmutation_sql, on='donor_id', how='left')
merged_df['genetic_mutation'] = merged_df['genetic_mutation'].fillna('unknown')
print("Core dataset shape after merge steps:", merged_df.shape)


# ==============================================================================
# STAGE 6: DONOR IDENTIFIER SWAPS AND CONTAMINATION/BATCH REMOVALS
# ==============================================================================
# Hand-curated swaps representing sender-receiver swaps (e.g. sample mixups)
pairs = [
    ("NND 55-AML", "NND 92-RDU"),  # ('sender', 'receiver')
    ("NND 10-HPE", "NND 53-BFD"),
    ("NND 68-MKZ", "NND 38-ICU"),
    ("NND 33-LCN", "NND 68-MKZ"),
    ("NND 25-NVD", "NND 33-LCN"),
    ("NND 35-CFC", "NND 50-TOM"),
    ("NND 86-OKC", "NND 14-PBK")
]

merged_df["original_donor_id"] = merged_df['donor_id'].copy()
merged_df["genetic_donor_id"] = merged_df['donor_id'].copy()
PCs = ["PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7", "PC8", "PC9", "PC10"]
all_cols = prs_interest + PCs

# Execute replacement algorithm to copy genotypes to recipient clinical profiles
for sender, receiver in pairs:
    updated_row = merged_df.loc[merged_df['donor_id'] == receiver].copy()
    updated_row['donor_id'] = f'{receiver}_from_{sender}'
    updated_row['genetic_donor_id'] = f'{sender}'
       
    sender_row_prsscores = merged_df.loc[merged_df['donor_id'] == sender, all_cols]    
    updated_row[all_cols] = sender_row_prsscores.values
    merged_df = pd.concat([merged_df, updated_row], ignore_index=True)

# Drop redundant sender ids, unresolvable mappings, and known mixed/corrupted batches
remove = [
    'NND 25-NVD', 'NND 10-HPE', 'NND 55-AML', 'NND 35-CFC', 'NND 86-OKC', 
    'NND 33-LCN', 'NND 68-MKZ', 'NND 38-ICU', 'NND 53-BFD', 'NND 92-RDU', 
    'NND 50-TOM', 'NND 14-PBK', 'NND 53-YFQ'
]
merged_df = merged_df[~merged_df['donor_id'].isin(remove)]

# Filter out pediatric or adolescent donors
merged_df = merged_df[merged_df["age"] >= 21]




# 2. Data Prep: APOE Effect Adjustment <a name="2-analysis-apoe-effect-adjustment"></a>


In [ ]:

# ==============================================================================
# STAGE 7: NORMALIZATION AND APOE VARIANT CORRECTION
# ==============================================================================
print("\n>>> Re-scaling features and modeling APOE parameters...")

# Normalize standard PRS scores using Z-score formulation: 
# $$z = \frac{x - \mu}{\sigma}$$
merged_df[prs_interest] = merged_df[prs_interest].apply(zscore)

# Load external APOE zygosity dose mappings
apoe = pd.read_csv(PATH_APOE_DATA, sep="\t")
apoe = apoe[["sample", "e2_dose", "e4_dose"]]
apoe["donor_id_apoe"] = "NND " + apoe["sample"]

# Perform direct genetic alignment
disorder_prs_apoe = pd.merge(
    merged_df, 
    apoe[["donor_id_apoe", "e2_dose", "e4_dose"]],
    left_on="genetic_donor_id",
    right_on="donor_id_apoe",
    how="inner"
)

# Linear combination of APOE component based on experimental beta coefficients:
# $$APOE_{component} = B_{e2} \cdot e2_{dose} + B_{e4} \cdot e4_{dose}$$
# Where: $B_{e2} = -0.47$ and $B_{e4} = 1.12$
B_e2 = -0.47
B_e4 = 1.12

disorder_prs_apoe["APOE_component"] = (
    B_e2 * disorder_prs_apoe["e2_dose"] +
    B_e4 * disorder_prs_apoe["e4_dose"]
)

# Calculate consolidated score:
# $$AD_{APOE} = ad\_without\_bigapoe\_0.01 + APOE_{component}$$
disorder_prs_apoe["AD_APOE"] = (
    disorder_prs_apoe["ad_without_bigapoe_0.01"] +
    disorder_prs_apoe["APOE_component"]
)

merged_df = merged_df.merge(
    disorder_prs_apoe[["donor_id", "AD_APOE", "e2_dose", "e4_dose"]], 
    on="donor_id", 
    how="left"
)

prs_interest2 = prs_interest.copy()
if "AD_APOE" not in prs_interest2:
    prs_interest2.insert(prs_interest2.index('ad_without_bigapoe_0.01') + 1, 'AD_APOE') 

# Construct traditional diploid genotypes based on e2 and e4 allele doses
save_apoe = merged_df[['donor_id', "e2_dose", "e4_dose"]].copy()
e2 = save_apoe["e2_dose"]
e4 = save_apoe["e4_dose"]

conditions = [
    (e2 == 2),
    (e4 == 2),
    (e2 == 1) & (e4 == 1),
    (e2 == 1) & (e4 == 0),
    (e2 == 0) & (e4 == 1),
    (e2 == 0) & (e4 == 0)
]
choices = ["e2e2", "e4e4", "e2e4", "e2e3", "e3e4", "e3e3"]
save_apoe["apoe_status"] = np.select(conditions, choices, default=np.nan)


# ==============================================================================
# STAGE 8: DIAGNOSTIC RE-CLASSIFICATION AND SAVE
# ==============================================================================
print("\n>>> Harmonizing clinical diagnosis nomenclature...")

def standardize_nd_current(nd_current):
    elements = sorted(nd_current.split(", "))
    return ", ".join(elements)

merged_df['nd_clean'] = merged_df['nd_current'].apply(standardize_nd_current)



# Simplify main category naming conventions
merged_df['nd_simple'] = merged_df['nd_clean'].replace(replace_dict_overview)
merged_df['nd_ftd'] = merged_df['nd_clean'].replace(replace_dict_ftd)

# (Optional) Save the final cleaned APOE & PRS models
save_apoe2 = merged_df[['donor_id', "original_donor_id", "genetic_donor_id", "e4_dose", "e2_dose", "ad_without_bigapoe_0.01", "AD_APOE", "AD", "nd_current", "nd_full_names"]].copy()
#save_apoe2.to_csv('NND_modeled_APOE_PRS.csv', sep='\t', index=False)

print("\n>>> Pipeline execution completed successfully! Saved modeled APOE dataset.")
print("Output preview:")
display(save_apoe2.head(10))

In [ ]:
# --- SAVING: Run this in the cell where 'merged_df' is finalized ---
# Recommended to save the data and load in the next step
merged_df.to_csv("output/merged_data.csv", index=False)
print("Data saved successfully to merged_data.csv")



# 3. Data prep: symptoms information

In [ ]:
import os
import numpy as np
import pandas as pd

# ==============================================================================
# 1. PREDEFINED VARIABLES & CONFIGURATION
# ==============================================================================

# --- File Paths ---
OUTPUT_DIR = "output"
OUTPUT_FILE_NAME = "coombes_df.csv"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE_NAME)

# --- Feature Pruning Configuration ---
COLS_TO_DROP = [
    'donor_id', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 
    'genetic_mutation', 'nd_clean', 'nd_ftd'
]

# ==============================================================================
# 2. REQUIRED DATA (INPUT)
# ==============================================================================
"""
REQUIRED INPUT DATA:
1. `merged_df`: Global master dataset containing genetic scores and age metadata.
2. `symptoms`: Diagnostic symptom logs mapping to 'original_donor_id'.
"""

#--- Optional Data Loading (Uncomment if starting fresh) ---
if 'merged_df' not in locals():
    merged_df = pd.read_csv("output/merged_data.csv")
if 'symptoms' not in locals():
    symptoms = pd.read_csv("output/symptoms_data.csv")


# ==============================================================================
# 3. PROCESSES
# ==============================================================================

# --- Process 1: Demographic Pruning ---
symptom_merged_df = merged_df.copy()
symptom_merged_df = symptom_merged_df.drop(columns=COLS_TO_DROP, errors='ignore')

# --- Process 2: Longitudinal Merging & Subsetting ---
merged_df_with_symptoms_before = symptom_merged_df.merge(
    symptoms, 
    left_on='original_donor_id', 
    right_on='donor_id', 
    how='left'
)

# Extract and subset cohorts validated prior to 2022
merged_df_with_symptoms = merged_df_with_symptoms_before[
    merged_df_with_symptoms_before['batch_symptom'] == 'pre2022'
].copy()

# --- Process 3: Dynamic Age Calculation ---
merged_df_with_symptoms["age_at_symptom"] = np.where(
    merged_df_with_symptoms["year_of_report"] == -9,
    -9,
    merged_df_with_symptoms["age"] - (merged_df_with_symptoms["file_year"] - merged_df_with_symptoms["year_of_report"])
)


# ==============================================================================
# 4. OUTPUT: Save Preprocessed Dataset
# ==============================================================================

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

merged_df_with_symptoms.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Data preprocessed and written successfully to: {OUTPUT_PATH}")